In [1]:
import pandas as pd
import numpy as np
import keras
from keras import layers, ops
from sklearn.model_selection import KFold

import ast

In [2]:
def prepare_melodies_sliding(melodies, window_size=16, pitch_to_idx=None):
    """
    Create sliding-window training examples from melodies.

    Instead of truncating long melodies, each melody produces multiple
    overlapping windows. Every note in the corpus gets predicted exactly once.

    For predicting note at position t, the input is the window_size notes
    before it (left-padded if t < window_size).
    """
    if pitch_to_idx is None:
        all_pitches = sorted(set(p for mel in melodies for p in mel))
        pitch_to_idx = {p: i + 1 for i, p in enumerate(all_pitches)}

    idx_to_pitch = {i: p for p, i in pitch_to_idx.items()}
    vocab_size = max(pitch_to_idx.values()) + 1  # +1 for PAD at index 0
    PAD = 0

    xs = []
    ys = []
    skipped = 0

    for mel in melodies:
        indexed = [pitch_to_idx[p] for p in mel if p in pitch_to_idx]

        if len(indexed) < 2:
            skipped += 1
            continue

        for t in range(1, len(indexed)):
            # Context: up to window_size tokens before position t
            start = max(0, t - window_size)
            context = indexed[start:t]

            # Left-pad if context is shorter than window_size
            pad_len = window_size - len(context)
            inp = [PAD] * pad_len + context

            xs.append(inp)
            ys.append(indexed[t])

    if skipped > 0:
        print(f"  Warning: skipped {skipped} melodies (too short or unknown pitches)")

    print(f"  Total training examples: {len(xs)} (from {len(melodies)} melodies)")

    return (np.array(xs), np.array(ys),
            vocab_size, pitch_to_idx, idx_to_pitch)

In [4]:
# Model components
class TokenAndPositionEmbedding(layers.Layer):
    """Embeds tokens and adds learned positional encoding."""

    def __init__(self, max_len, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim, mask_zero=False
        )
        self.pos_emb = layers.Embedding(
            input_dim=max_len, output_dim=embed_dim
        )

    def call(self, x):
        seq_len = ops.shape(x)[-1]
        positions = ops.arange(start=0, stop=seq_len, step=1)
        token_embeddings = self.token_emb(x)
        position_embeddings = self.pos_emb(positions)
        return token_embeddings + position_embeddings


class SlidingWindowTransformerBlock(layers.Layer):
    """
    Transformer block where position i can only attend to
    positions [i-window_size+1, ..., i].
    """

    def __init__(self, embed_dim, num_heads, ff_dim, window_size, dropout_rate=0.1):
        super().__init__()
        self.window_size = window_size
        self.att = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim // num_heads
        )
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def _sliding_window_mask(self, seq_len):
        """Create a causal sliding window attention mask."""
        causal = ops.tril(ops.ones((seq_len, seq_len), dtype="bool"))
        rows = ops.arange(seq_len)[:, None]
        cols = ops.arange(seq_len)[None, :]
        window = (rows - cols) < self.window_size
        mask = ops.cast(causal & window, dtype="bool")
        return mask

    def call(self, inputs, training=False):
        seq_len = ops.shape(inputs)[1]
        mask = self._sliding_window_mask(seq_len)

        attn_output = self.att(
            inputs, inputs, attention_mask=mask
        )
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

In [5]:
def build_transformer(
    vocab_size,
    window_size=16,
    embed_dim=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout_rate=0.1,
):
    """
    Build a causal sliding-window transformer for next-pitch prediction.

    The window_size parameter is analogous to IDyOM's order bound:
    each position can only attend to the previous window_size positions.

    Default configuration is deliberately small for fair comparison with IDyOM:
    - embed_dim=32, num_heads=4, ff_dim=64, num_layers=2
    """
    inputs = keras.Input(shape=(window_size,))

    # Embedding
    x = TokenAndPositionEmbedding(window_size, vocab_size, embed_dim)(inputs)

    # Transformer blocks with sliding window causal masking
    for _ in range(num_layers):
        x = SlidingWindowTransformerBlock(
            embed_dim, num_heads, ff_dim, window_size, dropout_rate
        )(x)

    # Output: predict from last position only
    x = x[:, -1, :]  # (batch, embed_dim)
    outputs = layers.Dense(vocab_size, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

In [6]:
def compute_ic(model, xs, ys):
    """
    Compute Information Content (IC) in bits for each note prediction.

    Args:
        model: trained transformer model
        xs: input sequences (num_examples, window_size)
        ys: target tokens (num_examples,)

    Returns:
        mean_ic: mean IC in bits across all predicted notes
        all_ics: list of IC values per note
    """
    probs = model.predict(xs, verbose=0)  # (num_examples, vocab_size)

    all_ics = []
    for i in range(len(xs)):
        p = probs[i][ys[i]]
        p = max(p, 1e-10)
        all_ics.append(-np.log2(p))

    return np.mean(all_ics), all_ics


def compute_ic_per_melody(model, melodies, window_size, pitch_to_idx):
    """
    Compute mean IC per melody using sliding windows.

    Args:
        model: trained transformer model
        melodies: list of pitch sequences
        window_size: context window size
        pitch_to_idx: vocabulary mapping

    Returns:
        list of dicts with melody_id and mean_ic
    """
    PAD = 0
    melody_ics = []

    for mel_idx, mel in enumerate(melodies):
        indexed = [pitch_to_idx[p] for p in mel if p in pitch_to_idx]
        if len(indexed) < 2:
            continue

        mel_xs = []
        mel_ys = []

        for t in range(1, len(indexed)):
            start = max(0, t - window_size)
            context = indexed[start:t]
            pad_len = window_size - len(context)
            inp = [PAD] * pad_len + context

            mel_xs.append(inp)
            mel_ys.append(indexed[t])

        mel_xs = np.array(mel_xs)
        mel_ys = np.array(mel_ys)
        probs = model.predict(mel_xs, verbose=0)

        ics = []
        for i in range(len(mel_xs)):
            p = max(probs[i][mel_ys[i]], 1e-10)
            ics.append(-np.log2(p))

        melody_ics.append({
            "melody_idx": mel_idx,
            "mean_ic": np.mean(ics),
            "n_notes": len(ics),
        })

    return melody_ics

In [7]:
def train_model(
    melodies,
    window_size=10,
    embed_dim=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout_rate=0.1,
    batch_size=32,
    epochs=50,
    validation_split=0.1,
    patience=5,
):
    """
    Simple full-corpus training (for initial testing only).
    For proper evaluation, use run_kfold.
    """
    xs, ys, vocab_size, pitch_to_idx, idx_to_pitch = \
        prepare_melodies_sliding(melodies, window_size=window_size)

    print(f"Data prepared:")
    print(f"  Melodies: {len(melodies)}")
    print(f"  Vocab size: {vocab_size} ({vocab_size - 1} pitches + PAD)")
    print(f"  Window size: {window_size}")
    print(f"  Training examples: {xs.shape[0]}")

    model = build_transformer(
        vocab_size=vocab_size, window_size=window_size, embed_dim=embed_dim,
        num_heads=num_heads, ff_dim=ff_dim, num_layers=num_layers,
        dropout_rate=dropout_rate,
    )

    print(f"\n  Total parameters: {model.count_params():,}")
    model.summary()

    model.compile(
        loss=keras.losses.SparseCategoricalCrossentropy(),
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    )

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=patience, restore_best_weights=True,
    )

    history = model.fit(
        xs, ys,
        batch_size=batch_size, epochs=epochs,
        validation_split=validation_split, callbacks=[early_stop],
    )

    return model, history, {
        "vocab_size": vocab_size,
        "pitch_to_idx": pitch_to_idx,
        "idx_to_pitch": idx_to_pitch,
        "window_size": window_size,
    }

In [7]:
#     IDyOM command per fold:
# (idyom:idyom <test-dataset-id> '(cpitch) '(cpitch) :models :ltm :pretraining-ids '(<train-dataset-id>) :k 1)

def run_kfold(
    melodies,
    melody_ids=None,
    k=10,
    window_size=16,
    embed_dim=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout_rate=0.1,
    batch_size=32,
    epochs=50,
    patience=5,
    random_state=42,
    export_folds_path=None,
):
    """
    K-fold cross-validation for fair comparison with IDyOM.

    Both models must use identical train/test splits. This function:
    1. Splits melodies into k folds at the piece level
    2. For each fold, trains on training melodies, evaluates on test melodies
    3. Exports fold assignments so IDyOM can use the same splits

    The sliding window ensures no melody is truncated, and the context
    window matches IDyOM's order bound.

    The exported CSV has columns: melody_id, fold, split
    Use it to create matching IDyOM datasets for each fold.
    """
    if melody_ids is None:
        melody_ids = list(range(len(melodies)))

    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)

    all_ics = []
    fold_results = []
    fold_assignments = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(melodies)):
        print(f"\n{'='*60}")
        print(f"FOLD {fold + 1}/{k}")
        print(f"{'='*60}")

        train_melodies = [melodies[i] for i in train_idx]
        test_melodies = [melodies[i] for i in test_idx]
        train_ids = [melody_ids[i] for i in train_idx]
        test_ids = [melody_ids[i] for i in test_idx]

        print(f"Train: {len(train_melodies)} melodies")
        print(f"Test:  {len(test_melodies)} melodies")

        # Record fold assignments
        for idx in train_idx:
            fold_assignments.append({
                "melody_id": melody_ids[idx],
                "fold": fold + 1,
                "split": "train",
            })
        for idx in test_idx:
            fold_assignments.append({
                "melody_id": melody_ids[idx],
                "fold": fold + 1,
                "split": "test",
            })

        # Prepare data: vocabulary built from training set only
        xs_train, ys_train, vocab_size, pitch_to_idx, _ = \
            prepare_melodies_sliding(train_melodies, window_size=window_size)

        # Test data uses same vocabulary
        xs_test, ys_test, _, _, _ = \
            prepare_melodies_sliding(test_melodies, window_size=window_size, pitch_to_idx=pitch_to_idx)

        print(f"Vocab size: {vocab_size}")

        # Build and train
        model = build_transformer(
            vocab_size=vocab_size, window_size=window_size, embed_dim=embed_dim,
            num_heads=num_heads, ff_dim=ff_dim, num_layers=num_layers,
            dropout_rate=dropout_rate,
        )

        if fold == 0:
            print(f"Total parameters: {model.count_params():,}")

        model.compile(
            loss=keras.losses.SparseCategoricalCrossentropy(),
            optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        )

        early_stop = keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience, restore_best_weights=True
        )

        model.fit(
            xs_train, ys_train,
            batch_size=batch_size, epochs=epochs,
            validation_split=0.1,
            callbacks=[early_stop], verbose=1,
        )

        # Evaluate on test fold
        mean_ic, ics = compute_ic(model, xs_test, ys_test)
        melody_ics = compute_ic_per_melody(model, test_melodies, window_size, pitch_to_idx)

        print(f"\nFold {fold + 1} mean IC: {mean_ic:.3f} bits")
        all_ics.extend(ics)
        fold_results.append({
            "fold": fold + 1,
            "mean_ic": mean_ic,
            "train_size": len(train_melodies),
            "test_size": len(test_melodies),
            "melody_ics": melody_ics,
        })

    # Export fold assignments for IDyOM replication
    df_folds = pd.DataFrame(fold_assignments)
    if export_folds_path:
        df_folds.to_csv(export_folds_path, index=False)
        print(f"\nFold assignments saved to: {export_folds_path}")

    # Summary
    overall_mean_ic = np.mean(all_ics)
    overall_std_ic = np.std(all_ics)

    print(f"\n{'='*60}")
    print(f"OVERALL RESULTS ({k}-fold cross-validation)")
    print(f"{'='*60}")
    print(f"Mean IC: {overall_mean_ic:.3f} bits")
    print(f"Std IC:  {overall_std_ic:.3f} bits")
    for r in fold_results:
        print(f"  Fold {r['fold']}: {r['mean_ic']:.3f} bits ({r['test_size']} test melodies)")

    return {
        "overall_mean_ic": overall_mean_ic,
        "overall_std_ic": overall_std_ic,
        "all_ics": all_ics,
        "fold_results": fold_results,
        "fold_assignments": df_folds,
    }

## Essen

In [ ]:
essen_melodies = pd.read_csv("essen_unique_melodies.csv")["pitch"].apply(ast.literal_eval).tolist()

model, history, data_info = train_model(essen_melodies, window_size=16, epochs=10)

In [ ]:
# Compute IC
xs, ys, _, _, _ = prepare_melodies_sliding(essen_melodies, window_size=10)
mean_ic, all_ics = compute_ic(model, xs, ys)
print(f"\nMean IC: {mean_ic:.3f} bits")

In [9]:
essen_melodies = pd.read_csv("essen_unique_melodies.csv")["pitch"].apply(ast.literal_eval).tolist()
essen_melody_ids = pd.read_csv("essen_unique_melodies.csv")["melody_id"].tolist()

results = run_kfold(
    essen_melodies, melody_ids=essen_melody_ids, k=10,
    window_size=10, epochs=35,
    export_folds_path="10_essen_fold_assignments_sliding.csv",
)


FOLD 1/10
Train: 7382 melodies
Test:  821 melodies
  Total training examples: 382205 (from 7382 melodies)
  Total training examples: 42279 (from 821 melodies)
Vocab size: 52
Total parameters: 20,788
Epoch 1/35
10750/10750 ━━━━━━━━━━━━━━━━━━━━ 58s 4ms/step - loss: 1.8858 - val_loss: 1.8489
Epoch 2/35
10750/10750 ━━━━━━━━━━━━━━━━━━━━ 31s 3ms/step - loss: 1.8065 - val_loss: 1.8277
Epoch 3/35
10750/10750 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - loss: 1.7875 - val_loss: 1.8245
Epoch 4/35
10750/10750 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - loss: 1.7776 - val_loss: 1.8128
Epoch 5/35
10750/10750 ━━━━━━━━━━━━━━━━━━━━ 29s 3ms/step - loss: 1.7702 - val_loss: 1.8170
Epoch 6/35
10750/10750 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - loss: 1.7633 - val_loss: 1.8124
Epoch 7/35
10750/10750 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - loss: 1.7591 - val_loss: 1.8135
Epoch 8/35
10750/10750 ━━━━━━━━━━━━━━━━━━━━ 29s 3ms/step - loss: 1.7549 - val_loss: 1.8142
Epoch 9/35
10750/10750 ━━━━━━━━━━━━━━━━━━━━ 29s 3ms/step - loss: 1.7521 

## Meertens

In [10]:
meertens_melodies = pd.read_csv("meertens_unique_melodies.csv")["pitch"].apply(ast.literal_eval).tolist()
meertens_melody_ids = pd.read_csv("meertens_unique_melodies.csv")["melody_id"].tolist()

In [ ]:
meertens_model, meertens_history, meertens_data_info = train_model(
    meertens_melodies,
    window_size=16,
    epochs=50
)

In [ ]:
# Compute IC
xs, ys, _, _, _ = prepare_melodies_sliding(meertens_melodies, window_size=16)
mean_ic, all_ics = compute_ic(meertens_model, xs, ys)
print(f"\nMean IC: {mean_ic:.3f} bits")

In [12]:
results = run_kfold(
    meertens_melodies, melody_ids=meertens_melody_ids, k=10,
    window_size=10, epochs=75,
    export_folds_path="meertens_10_fold_assignments.csv",
)


FOLD 1/10
Train: 15858 melodies
Test:  1762 melodies
  Total training examples: 1128111 (from 15858 melodies)
  Total training examples: 124155 (from 1762 melodies)
Vocab size: 46
Total parameters: 20,398
Epoch 1/75
31729/31729 ━━━━━━━━━━━━━━━━━━━━ 115s 3ms/step - loss: 1.8146 - val_loss: 1.7502
Epoch 2/75
31729/31729 ━━━━━━━━━━━━━━━━━━━━ 93s 3ms/step - loss: 1.7582 - val_loss: 1.7363
Epoch 3/75
31729/31729 ━━━━━━━━━━━━━━━━━━━━ 93s 3ms/step - loss: 1.7457 - val_loss: 1.7271
Epoch 4/75
31729/31729 ━━━━━━━━━━━━━━━━━━━━ 93s 3ms/step - loss: 1.7389 - val_loss: 1.7171
Epoch 5/75
31729/31729 ━━━━━━━━━━━━━━━━━━━━ 93s 3ms/step - loss: 1.7337 - val_loss: 1.7161
Epoch 6/75
31729/31729 ━━━━━━━━━━━━━━━━━━━━ 94s 3ms/step - loss: 1.7301 - val_loss: 1.7185
Epoch 7/75
31729/31729 ━━━━━━━━━━━━━━━━━━━━ 94s 3ms/step - loss: 1.7279 - val_loss: 1.7106
Epoch 8/75
31729/31729 ━━━━━━━━━━━━━━━━━━━━ 93s 3ms/step - loss: 1.7260 - val_loss: 1.7134
Epoch 9/75
31729/31729 ━━━━━━━━━━━━━━━━━━━━ 93s 3ms/step - loss: 

## Sliding window Transformer with Intervals feature

In [31]:
def get_fixed_interval_params():
    """
    Fixed interval vocabulary covering [-48, +48] semitones (4 octaves).

    This avoids any data leakage between folds and matches IDyOM's
    approach where the viewpoint alphabet is defined by the domain,
    not by the data.
    """
    min_interval = -48
    max_interval = 48
    interval_offset = -min_interval + 1
    interval_vocab_size = max_interval - min_interval + 2

    return interval_offset, interval_vocab_size


def prepare_melodies_intervals(melodies, window_size=10, pitch_to_idx=None,
                            interval_offset=None, interval_vocab_size=None):
    """Prepare sliding windows with pitch AND interval inputs."""
    if pitch_to_idx is None:
        all_pitches = sorted(set(p for mel in melodies for p in mel))
        pitch_to_idx = {p: i + 1 for i, p in enumerate(all_pitches)}

    idx_to_pitch = {i: p for p, i in pitch_to_idx.items()}
    vocab_size = max(pitch_to_idx.values()) + 1
    PAD = 0

    # If interval stats not provided, compute from data given
    if interval_offset is None or interval_vocab_size is None:
        interval_offset, interval_vocab_size = get_fixed_interval_params()

    xs_pitch = []
    xs_interval = []
    ys = []
    skipped = 0

    for mel in melodies:
        indexed = [pitch_to_idx[p] for p in mel if p in pitch_to_idx]
        raw = [p for p in mel if p in pitch_to_idx]

        if len(indexed) < 2:
            skipped += 1
            continue

        for t in range(1, len(indexed)):
            # Up to window_size tokens before position t
            start = max(0, t - window_size)
            context = indexed[start:t]
            raw_context = raw[start:t]

            # Compute raw intervals (0 for the first context note)
            raw_intervals = [0]
            for i in range(1, len(raw_context)):
                raw_intervals.append(raw_context[i] - raw_context[i - 1])

            # Left-pad if context is shorter than window_size
            pad_len = window_size - len(context)

            pitch_seq = [PAD] * pad_len + context

            # Shift intervals to non-negative indices, clamp to valid range
            interval_seq = [PAD] * pad_len
            for iv in raw_intervals:
                shifted = iv + interval_offset
                # Clamp to valid range [1, interval_vocab_size - 1]
                shifted = max(1, min(shifted, interval_vocab_size - 1))
                interval_seq.append(shifted)

            xs_pitch.append(pitch_seq)
            xs_interval.append(interval_seq)
            ys.append(indexed[t])

    if skipped > 0:
        print(f"  Warning: skipped {skipped} melodies (too short or unknown pitches)")

    return (np.array(xs_pitch), np.array(xs_interval), np.array(ys),
            vocab_size, pitch_to_idx, idx_to_pitch)

In [32]:
def build_transformer_intervals(
    vocab_size,
    interval_vocab_size,
    window_size=10,
    embed_dim=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout_rate=0.1,
):
    """
    Build a causal sliding-window transformer for next-pitch prediction
    with both pitch and interval inputs.
    """
    pitch_input = keras.Input(shape=(window_size,), name="pitch_input")
    interval_input = keras.Input(shape=(window_size,), name="interval_input")

    # Pitch embedding with positional encoding
    pitch_emb = TokenAndPositionEmbedding(window_size, vocab_size, embed_dim)(pitch_input)

    # Interval embedding
    interval_emb = layers.Embedding(
        input_dim=interval_vocab_size, output_dim=embed_dim, mask_zero=False
    )(interval_input)

    # Add pitch and interval representations
    x = layers.Add()([pitch_emb, interval_emb])

    # Transformer blocks with sliding window causal masking
    for _ in range(num_layers):
        x = SlidingWindowTransformerBlock(
            embed_dim, num_heads, ff_dim, window_size, dropout_rate
        )(x)

    # Output: predict from last position only
    x = x[:, -1, :]
    outputs = layers.Dense(vocab_size, activation="softmax")(x)

    model = keras.Model(inputs=[pitch_input, interval_input], outputs=outputs)
    return model

In [33]:
def compute_ic_intervals(model, xs_pitch, xs_interval, ys):
    """
    Compute Information Content (IC) in bits for each note prediction.
    Both pitch and interval input arrays are passed to them model.
    """
    probs = model.predict([xs_pitch, xs_interval], verbose=0)

    all_ics = []
    for i in range(len(xs_pitch)):
        p = probs[i][ys[i]]
        p = max(p, 1e-10)
        all_ics.append(-np.log2(p))

    return np.mean(all_ics), all_ics


def compute_ic_per_melody_intervals(model, melodies, window_size, pitch_to_idx,
                          interval_offset, interval_vocab_size):
    """
    Compute mean IC per melody using sliding windows.
    Both pitch and interval input arrays are passed to them model.
    """
    PAD = 0
    melody_ics = []

    for mel_idx, mel in enumerate(melodies):
        indexed = [pitch_to_idx[p] for p in mel if p in pitch_to_idx]
        raw = [p for p in mel if p in pitch_to_idx]
        if len(indexed) < 2:
            continue

        mel_xs_pitch = []
        mel_xs_interval = []
        mel_ys = []

        for t in range(1, len(indexed)):
            start = max(0, t - window_size)
            context = indexed[start:t]
            raw_context = raw[start:t]

            # Compute intervals
            raw_intervals = [0]
            for i in range(1, len(raw_context)):
                raw_intervals.append(raw_context[i] - raw_context[i - 1])

            pad_len = window_size - len(context)

            pitch_seq = [PAD] * pad_len + context
            interval_seq = [PAD] * pad_len
            for iv in raw_intervals:
                shifted = iv + interval_offset
                shifted = max(1, min(shifted, interval_vocab_size - 1))
                interval_seq.append(shifted)

            mel_xs_pitch.append(pitch_seq)
            mel_xs_interval.append(interval_seq)
            mel_ys.append(indexed[t])

        mel_xs_pitch = np.array(mel_xs_pitch)
        mel_xs_interval = np.array(mel_xs_interval)
        mel_ys = np.array(mel_ys)
        probs = model.predict([mel_xs_pitch, mel_xs_interval], verbose=0)

        ics = []
        for i in range(len(mel_xs_pitch)):
            p = max(probs[i][mel_ys[i]], 1e-10)
            ics.append(-np.log2(p))

        melody_ics.append({
            "melody_idx": mel_idx,
            "mean_ic": np.mean(ics),
            "n_notes": len(ics),
        })

    return melody_ics

In [36]:
def run_kfold_intervals(
    melodies,
    melody_ids=None,
    k=10,
    window_size=10,
    embed_dim=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout_rate=0.1,
    batch_size=32,
    epochs=50,
    patience=5,
    random_state=42,
    export_folds_path=None,
):
    """
    K-fold cross-validation for fair comparison with IDyOM.

    Both models must use identical train/test splits. This function:
    1. Splits melodies into k folds at the piece level
    2. For each fold, trains on training melodies, evaluates on test melodies
    3. Exports fold assignments so IDyOM can use the same splits

    The exported CSV has columns: melody_id, fold, split
    Use it to create matching IDyOM datasets for each fold.
    """
    if melody_ids is None:
        melody_ids = list(range(len(melodies)))

    # Compute global interval stats BEFORE splitting
    # Ensures train and test folds use the same interval vocabulary
    print("Computing global interval statistics...")
    interval_offset, interval_vocab_size = get_fixed_interval_params()

    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)

    all_ics = []
    fold_results = []
    fold_assignments = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(melodies)):
        print(f"\n{'='*60}")
        print(f"FOLD {fold + 1}/{k}")
        print(f"{'='*60}")

        train_melodies = [melodies[i] for i in train_idx]
        test_melodies = [melodies[i] for i in test_idx]
        train_ids = [melody_ids[i] for i in train_idx]
        test_ids = [melody_ids[i] for i in test_idx]

        print(f"Train: {len(train_melodies)} melodies")
        print(f"Test:  {len(test_melodies)} melodies")

        # Record fold assignments
        for idx in train_idx:
            fold_assignments.append({
                "melody_id": melody_ids[idx],
                "fold": fold + 1,
                "split": "train",
            })
        for idx in test_idx:
            fold_assignments.append({
                "melody_id": melody_ids[idx],
                "fold": fold + 1,
                "split": "test",
            })

        # Prepare training data
        xs_pitch_train, xs_int_train, ys_train, vocab_size, pitch_to_idx, _ = \
            prepare_melodies_intervals(
                train_melodies, window_size=window_size,
                interval_offset=interval_offset, interval_vocab_size=interval_vocab_size,
            )

        # Test data
        xs_pitch_test, xs_int_test, ys_test, _, _, _ = \
            prepare_melodies_intervals(
                test_melodies, window_size=window_size,
                pitch_to_idx=pitch_to_idx,
                interval_offset=interval_offset, interval_vocab_size=interval_vocab_size,
            )

        print(f"Vocab size: {vocab_size}")
        print(f"Interval vocab size: {interval_vocab_size}")

        # Build and train
        model = build_transformer_intervals(
            vocab_size=vocab_size, interval_vocab_size=interval_vocab_size,
            window_size=window_size, embed_dim=embed_dim,
            num_heads=num_heads, ff_dim=ff_dim, num_layers=num_layers,
            dropout_rate=dropout_rate,
        )

        if fold == 0:
            print(f"Total parameters: {model.count_params():,}")

        model.compile(
            loss=keras.losses.SparseCategoricalCrossentropy(),
            optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        )

        early_stop = keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience, restore_best_weights=True
        )

        model.fit(
            [xs_pitch_train, xs_int_train], ys_train,
            batch_size=batch_size, epochs=epochs,
            validation_split=0.1,
            callbacks=[early_stop], verbose=1,
        )

        # Evaluate on test fold
        mean_ic, ics = compute_ic_intervals(model, xs_pitch_test, xs_int_test, ys_test)
        melody_ics = compute_ic_per_melody_intervals(
            model, test_melodies, window_size, pitch_to_idx,
            interval_offset, interval_vocab_size,
        )

        print(f"\nFold {fold + 1} mean IC: {mean_ic:.3f} bits")
        all_ics.extend(ics)
        fold_results.append({
            "fold": fold + 1,
            "mean_ic": mean_ic,
            "train_size": len(train_melodies),
            "test_size": len(test_melodies),
            "melody_ics": melody_ics,
        })

    # Export fold assignments for IDyOM replication
    df_folds = pd.DataFrame(fold_assignments)
    if export_folds_path:
        df_folds.to_csv(export_folds_path, index=False)
        print(f"\nFold assignments saved to: {export_folds_path}")

    # Summary
    overall_mean_ic = np.mean(all_ics)
    overall_std_ic = np.std(all_ics)

    print(f"\n{'='*60}")
    print(f"OVERALL RESULTS ({k}-fold cross-validation)")
    print(f"{'='*60}")
    print(f"Mean IC: {overall_mean_ic:.3f} bits")
    print(f"Std IC:  {overall_std_ic:.3f} bits")
    for r in fold_results:
        print(f"  Fold {r['fold']}: {r['mean_ic']:.3f} bits ({r['test_size']} test melodies)")

    return {
        "overall_mean_ic": overall_mean_ic,
        "overall_std_ic": overall_std_ic,
        "all_ics": all_ics,
        "fold_results": fold_results,
        "fold_assignments": df_folds,
    }

In [37]:
essen_melodies = pd.read_csv("essen_unique_melodies.csv")["pitch"].apply(ast.literal_eval).tolist()
essen_melody_ids = pd.read_csv("essen_unique_melodies.csv")["melody_id"].tolist()

results_intervals = run_kfold_intervals(
    essen_melodies, melody_ids=essen_melody_ids, k=10,
    window_size=10, epochs=35,
    export_folds_path="intervals_10_essen_fold_assignments_sliding.csv",
)

Computing global interval statistics...

FOLD 1/10
Train: 7382 melodies
Test:  821 melodies
Vocab size: 52
Interval vocab size: 98
Total parameters: 23,924
Epoch 1/35
10750/10750 ━━━━━━━━━━━━━━━━━━━━ 51s 4ms/step - loss: 1.8711 - val_loss: 1.8410
Epoch 2/35
10750/10750 ━━━━━━━━━━━━━━━━━━━━ 31s 3ms/step - loss: 1.7880 - val_loss: 1.8381
Epoch 3/35
10750/10750 ━━━━━━━━━━━━━━━━━━━━ 31s 3ms/step - loss: 1.7685 - val_loss: 1.8238
Epoch 4/35
10750/10750 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - loss: 1.7562 - val_loss: 1.8098
Epoch 5/35
10750/10750 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - loss: 1.7485 - val_loss: 1.8005
Epoch 6/35
10750/10750 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - loss: 1.7425 - val_loss: 1.8198
Epoch 7/35
10750/10750 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - loss: 1.7364 - val_loss: 1.7940
Epoch 8/35
10750/10750 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - loss: 1.7322 - val_loss: 1.8124
Epoch 9/35
10750/10750 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - loss: 1.7294 - val_loss: 1.7909
Epoch 10/35
10750/10750 ━